# Week 18 · Notebook 01: Foundry Serverless Endpoints

# Requirements: pip install openai azure-ai-projects azure-identity

# ⚠️ REQUIRES: Azure subscription + AI Foundry project

> 💰 COST WARNING: set a budget alert before running, serverless endpoints bill per token and nothing here is free by default.


## What you build

Deploy a serverless model, call it two ways (the OpenAI SDK and `azure-ai-projects`), then run the Week 6 bill-of-lading extraction prompt against a Foundry model and measure per-field accuracy and estimated cost. Every cloud call is wrapped so missing credentials print setup instructions instead of crashing, the notebook runs in **dry-run** mode without Azure.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (template)
# Robust fallback: walk up until we find zoro/data.py, in case Jupyter started elsewhere.
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro" / "data.py").exists():
        sys.path.insert(0, str(_p))
        break

import os
import json
import numpy as np
import pandas as pd
from zoro import data

SEED = 18
rng = np.random.default_rng(SEED)
print("zoro.data ready; seed =", SEED)


## Serverless endpoints (Models-as-a-Service)

A serverless endpoint is a pay-per-token, Microsoft-managed deployment of a catalog model. You deploy a *name* and call an HTTPS URL, no compute to own. In Azure, `model=` is your **deployment name**, not the raw model id: the classic Week 18 gotcha.

**Deploy one first** (portal: Model catalog → pick a model → Deploy → Serverless API endpoint), or via CLI:

```bash
az login
az extension add --name ml
# create hub + project, then deploy from the catalog and copy the endpoint URL / key
```

Then point the environment variables below at your deployment.


In [ ]:
bols = data.bol_samples(n=8, seed=5)  # 8 synthetic bills of lading with ground truth

EXTRACT_PROMPT = (
    "Extract the bill-of-lading fields from the text below.\n"
    "Return ONLY valid JSON with exactly these keys:\n"
    "shipper, consignee, port_of_loading, port_of_discharge, commodity,\n"
    "quantity, gross_weight_kg, declared_value_usd, freight_terms, date_of_issue.\n"
    "quantity and gross_weight_kg must be integers; declared_value_usd a number.\n\n"
    "TEXT:\n{text}"
)

print("Loaded", len(bols), "BoL samples; first id:", bols[0]["bol_id"])


## Authentication

Preferred: Entra ID via `DefaultAzureCredential`. Quick start: API key. We read everything from environment variables and never hardcode a key. The two env-var shapes map to the two call paths below.


In [ ]:
conn_str = os.environ.get("AZURE_AI_PROJECT_CONNECTION_STRING")
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
api_key = os.environ.get("AZURE_OPENAI_API_KEY")

AZURE_READY = bool(conn_str or (endpoint and api_key))

if not AZURE_READY:
    print("⚠️ Azure credentials not found, running in DRY-RUN mode.")
    print("Set one of the following, then restart the kernel:")
    print(" export AZURE_AI_PROJECT_CONNECTION_STRING='<project connection string>' (azure-ai-projects)")
    print(" export AZURE_OPENAI_ENDPOINT='https://<resource>.openai.azure.com' (OpenAI SDK)")
    print(" export AZURE_OPENAI_API_KEY='<your key>'")
else:
    print("✅ Azure credentials found.")


## Call path 1: OpenAI SDK against Azure OpenAI

The most common quick start. The `model=` argument is your **deployment name**.


In [ ]:
def call_openai_sdk(prompt, deployment="gpt-4.1"):
    from openai import AzureOpenAI
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-10-21",
    )
    resp = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return resp.choices[0].message.content

if AZURE_READY and endpoint and api_key:
    try:
        out = call_openai_sdk("Reply with exactly: OK")
        print("OpenAI SDK call returned:", out[:60])
    except Exception as e:  # noqa: BLE001
        print("OpenAI SDK call failed (check deployment name + endpoint):", type(e).__name__, e)
else:
    print("DRY-RUN: skipped OpenAI SDK call (set AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY).")


## Call path 2: azure-ai-projects (Foundry-native)

`AIProjectClient` is the Foundry-native entry point with typed access to agents, evals, and tracing. It needs the project connection string plus an Entra credential. The exact inference method name varies by SDK version, see the client-library docs in the Sources.


In [ ]:
def call_ai_projects(prompt):
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential
    client = AIProjectClient.from_connection_string(
        conn_str=conn_str,
        credential=DefaultAzureCredential(),
    )
    resp = client.inference.get_chat_completions(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content

if AZURE_READY and conn_str:
    try:
        out = call_ai_projects("Reply with exactly: OK")
        print("azure-ai-projects call returned:", out[:60])
    except Exception as e:  # noqa: BLE001
        print("azure-ai-projects call failed (SDK version may differ):", type(e).__name__, e)
else:
    print("DRY-RUN: skipped azure-ai-projects call (set AZURE_AI_PROJECT_CONNECTION_STRING).")


## Run the Week 6 BoL extraction prompt suite

For each synthetic BoL, call the model, parse the JSON, and compare every field against ground truth. Per-field accuracy is the metric, pick it before the model, as always. Keep the cloud run small (4 docs) until you confirm cost.


In [ ]:
def normalize_string(s):
    return str(s).strip().lower()

def field_matches(pred, truth, key):
    if pred is None or truth is None:
        return False
    if key in ("quantity", "gross_weight_kg"):
        try:
            return int(float(pred)) == int(float(truth))
        except (TypeError, ValueError):
            return False
    if key == "declared_value_usd":
        try:
            return abs(float(pred) - float(truth)) < 0.01
        except (TypeError, ValueError):
            return False
    return normalize_string(pred) == normalize_string(truth)

FIELDS = ["shipper", "consignee", "port_of_loading", "port_of_discharge",
          "commodity", "quantity", "gross_weight_kg", "declared_value_usd",
          "freight_terms", "date_of_issue"]

predictions = []
usage_tokens = {"in": 0, "out": 0}

if AZURE_READY:
    for b in bols[:4]:  # raise to 8 when confident about cost
        prompt = EXTRACT_PROMPT.format(text=b["text"])
        try:
            raw = call_openai_sdk(prompt) if (endpoint and api_key) else call_ai_projects(prompt)
            raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            pred = json.loads(raw)
        except Exception as e:  # noqa: BLE001
            pred = None
            raw = ""
            print("  failed on", b["bol_id"], "->", type(e).__name__)
        predictions.append((b, pred))
        usage_tokens["in"] += len(prompt) // 4
        usage_tokens["out"] += len(raw) // 4
else:
    predictions = [(b, None) for b in bols[:4]]
    print("DRY-RUN: skipped cloud extraction (no Azure credentials).")

print("Collected predictions for", len(predictions), "documents.")


In [ ]:
per_field = {f: {"correct": 0, "total": 0} for f in FIELDS}
for b, pred in predictions:
    truth = b["fields"]
    for f in FIELDS:
        per_field[f]["total"] += 1
        if pred and field_matches(pred.get(f), truth.get(f), f):
            per_field[f]["correct"] += 1

rows = []
for f in FIELDS:
    c, t = per_field[f]["correct"], per_field[f]["total"]
    rows.append({"field": f, "correct": c, "total": t, "accuracy": (c / t) if t else 0.0})

acc_df = pd.DataFrame(rows)
overall_acc = acc_df["correct"].sum() / acc_df["total"].sum() if acc_df["total"].sum() else 0.0
print(acc_df.to_string(index=False))
print(f"\nOVERALL_FIELD_ACCURACY: {overall_acc:.3f}")


## Cost printout

Serverless endpoints bill per token (input/output, priced separately). We estimate from character counts (≈ 4 chars/token) and a placeholder price you must verify against the live Azure pricing page, treat the number as an order of magnitude, not an invoice.


In [ ]:
# Placeholder prices per 1M tokens, VERIFY against live Azure OpenAI pricing.
PRICE_PER_1M_IN = 0.15 # USD
PRICE_PER_1M_OUT = 0.60 # USD

tokens_in = max(usage_tokens["in"], 1)
tokens_out = max(usage_tokens["out"], 1)
cost = (tokens_in * PRICE_PER_1M_IN + tokens_out * PRICE_PER_1M_OUT) / 1_000_000

print(f"Estimated tokens, in: {tokens_in:,} · out: {tokens_out:,}")
print(f"Estimated cost for this run: ${cost:.6f} USD")

TOTAL_ESTIMATED_COST_USD = cost
print(f"TOTAL_ESTIMATED_COST_USD: {TOTAL_ESTIMATED_COST_USD:.6f}")
